# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [173]:
import pandas as pd
import numpy as np
np.bool = bool

# For preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer # For handling missing values if they exist

# For models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# For model selection and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer # make_scorer for custom RMSE

# For explainability
import shap

In [174]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [175]:
# Get X and Y

# Define the target variable as outlined by the assignment objective
TARGET = 'area'

# Create the target data (Y)
Y = fires_dt[TARGET] 

# Create the features data frame (X)
# Drop the target column from a copy of the DataFrame to create X
X = fires_dt.drop(columns=[TARGET]) 

print("Features (X) head:")
print(X.head())
print("\nTarget (Y) head:")
print(Y.head())

# Identify numerical and categorical features for preprocessing later
numeric_features = X.select_dtypes(include=np.number).columns.tolist() 
categorical_features = X.select_dtypes(include='object').columns.tolist() 

print(f"\nNumerical Features: {numeric_features}")
print(f"Categorical Features: {categorical_features}")

Features (X) head:
   coord_x  coord_y month  day  ffmc   dmc     dc  isi  temp  rh  wind  rain
0        7        5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0
1        7        4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0
2        7        4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0
3        8        6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2
4        8        6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0

Target (Y) head:
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: area, dtype: float64

Numerical Features: ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
Categorical Features: ['month', 'day']


In [176]:
# Splitting strategy (testing) before scaling in the next question
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42) #using common defaults for test size and random state
print(f"Training set shape: {X_train.shape}, {Y_train.shape}")
print(f"Test set shape: {X_test.shape}, {Y_test.shape}")

Training set shape: (413, 12), (413,)
Test set shape: (104, 12), (104,)


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [177]:
# Preproc 1
numeric_transformer_p1 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Impute missing numeric values with the mean
    ('scaler', StandardScaler()) # Scale numeric variables
])

categorical_transformer_p1 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing categorical values with the most frequent
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # One-hot encode categorical variables
])

preproc1 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_p1, numeric_features),
        ('cat', categorical_transformer_p1, categorical_features)
    ])

print("Preproc 1 created.")

Preproc 1 created.


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [178]:
# Preproc 2

# Numeric transformer for preproc2: Impute -> Polynomial Features -> Scale
numeric_transformer_p2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)), # Applies a 2nd-degree polynomial transform
    ('scaler', StandardScaler())
])

# Categorical transformer remains the same as preproc1
categorical_transformer_p2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers into preproc2
preproc2 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_p2, numeric_features),
        ('cat', categorical_transformer_p2, categorical_features)
    ])

print("Preproc 2 created (using PolynomialFeatures for non-linear transformation).")

Preproc 2 created (using PolynomialFeatures for non-linear transformation).


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [179]:
# Pipeline A = preproc1 + baseline
# Define baseline and advanced regressors
baseline_regressor = KNeighborsRegressor()
advanced_regressor = RandomForestRegressor(random_state=42) # Set random_state for reproducibility, using randomforest because google said its best

# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline(steps=[('preprocessor', preproc1),
                             ('regressor', baseline_regressor)])
print("Pipeline A (preproc1 + KNeighborsRegressor) created.")

Pipeline A (preproc1 + KNeighborsRegressor) created.


In [180]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline(steps=[('preprocessor', preproc2),
                             ('regressor', baseline_regressor)])
print("Pipeline B (preproc2 + KNeighborsRegressor) created.")


Pipeline B (preproc2 + KNeighborsRegressor) created.


In [181]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline(steps=[('preprocessor', preproc1),
                             ('regressor', advanced_regressor)])
print("Pipeline C (preproc1 + RandomForestRegressor) created.")

Pipeline C (preproc1 + RandomForestRegressor) created.


In [182]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline(steps=[('preprocessor', preproc2),
                             ('regressor', advanced_regressor)])
print("Pipeline D (preproc2 + RandomForestRegressor) created.")
    

Pipeline D (preproc2 + RandomForestRegressor) created.


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [183]:
# Tune Hyperparams

# Define RMSE scorer
rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False, squared=False)

In [184]:
# --- Pipeline A Tuning ---
# 'regressor__n_neighbors': [3, 5, 7, 9] will give 4 combinations
param_grid_A = {
    'regressor__n_neighbors': [3, 5, 7, 9], # Tune 'n_neighbors'
}
grid_search_A = GridSearchCV(pipeline_A, param_grid_A, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring=rmse_scorer, n_jobs=-1, verbose=1)
grid_search_A.fit(X_train, Y_train)
print(f"Pipeline A Best Params: {grid_search_A.best_params_}")
print(f"Pipeline A Best CV RMSE: {-grid_search_A.best_score_:.4f}")

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Pipeline A Best Params: {'regressor__n_neighbors': 9}
Pipeline A Best CV RMSE: 43.3950


In [185]:
# --- Pipeline B Tuning ---
param_grid_B = {
    'preprocessor__num__poly__degree': [1, 2, 3], # Tune polynomial degree
    'regressor__n_neighbors': [5, 10, 15] # Tune n_neighbors
}
# This provides 3 * 3 = 9 combinations, satisfying the requirement.
grid_search_B = GridSearchCV(pipeline_B, param_grid_B, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring=rmse_scorer, n_jobs=-1, verbose=1)
grid_search_B.fit(X_train, Y_train)
print(f"Pipeline B Best Params: {grid_search_B.best_params_}")
print(f"Pipeline B Best CV RMSE: {-grid_search_B.best_score_:.4f}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Pipeline B Best Params: {'preprocessor__num__poly__degree': 2, 'regressor__n_neighbors': 15}
Pipeline B Best CV RMSE: 39.9912


In [186]:
# --- Pipeline C Tuning ---
# Hyperparameters for RandomForestRegressor in Pipeline C
param_grid_C = {
    'regressor__n_estimators': [50, 100, 200], # Number of trees
    'regressor__max_depth': [None, 10, 20] # Max depth of trees
}
# This provides 3 * 3 = 9 combinations.
grid_search_C = GridSearchCV(pipeline_C, param_grid_C, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring=rmse_scorer, n_jobs=-1, verbose=1)
grid_search_C.fit(X_train, Y_train)
print(f"Pipeline C Best Params: {grid_search_C.best_params_}")
print(f"Pipeline C Best CV RMSE: {-grid_search_C.best_score_:.4f}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Pipeline C Best Params: {'regressor__max_depth': None, 'regressor__n_estimators': 50}
Pipeline C Best CV RMSE: 49.8856


In [187]:
# --- Pipeline D Tuning ---
# Hyperparameters for RandomForestRegressor in Pipeline D (with PolynomialFeatures)
param_grid_D = {
    'preprocessor__num__poly__degree': [1, 2], # Tune polynomial degree
    'regressor__n_estimators': [100, 200], # Number of trees
    'regressor__max_depth': [10, 20] # Max depth of trees
}
# This provides 2 * 2 * 2 = 8 combinations.
grid_search_D = GridSearchCV(pipeline_D, param_grid_D, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring=rmse_scorer, n_jobs=-1, verbose=1)
grid_search_D.fit(X_train, Y_train)
print(f"Pipeline D Best Params: {grid_search_D.best_params_}")
print(f"Pipeline D Best CV RMSE: {-grid_search_D.best_score_:.4f}")

# Store best estimators and their scores for easy comparison
results = {
    'Pipeline A': {'best_estimator': grid_search_A.best_estimator_, 'best_cv_rmse': -grid_search_A.best_score_},
    'Pipeline B': {'best_estimator': grid_search_B.best_estimator_, 'best_cv_rmse': -grid_search_B.best_score_},
    'Pipeline C': {'best_estimator': grid_search_C.best_estimator_, 'best_cv_rmse': -grid_search_C.best_score_},
    'Pipeline D': {'best_estimator': grid_search_D.best_estimator_, 'best_cv_rmse': -grid_search_D.best_score_},
}

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Pipeline D Best Params: {'preprocessor__num__poly__degree': 1, 'regressor__max_depth': 20, 'regressor__n_estimators': 200}
Pipeline D Best CV RMSE: 51.4202


# Evaluate

+ Which model has the best performance?

In [188]:
# Evaluate

best_model_name = None
min_rmse = float('inf')

print("\n--- Model Performance on Test Set ---")
for name, res in results.items():
    model = res['best_estimator']
    test_predictions = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_predictions))
    test_mae = mean_absolute_error(Y_test, test_predictions)
    test_r2 = r2_score(Y_test, test_predictions)

    print(f"\n{name}:")
    print(f"  Best CV RMSE: {res['best_cv_rmse']:.4f}")
    print(f"  Test RMSE: {test_rmse:.4f}")
    print(f"  Test MAE: {test_mae:.4f}")
    print(f"  Test R-squared: {test_r2:.4f}")

    if test_rmse < min_rmse:
        min_rmse = test_rmse
        best_model_name = name

print(f"\n--- Conclusion ---")
print(f"The best performing model on the test set is: {best_model_name} with an RMSE of {min_rmse:.4f}")

# Assign the best performing model for later use
best_overall_model = results[best_model_name]['best_estimator']


--- Model Performance on Test Set ---

Pipeline A:
  Best CV RMSE: 43.3950
  Test RMSE: 108.8739
  Test MAE: 24.3682
  Test R-squared: -0.0056

Pipeline B:
  Best CV RMSE: 39.9912
  Test RMSE: 108.8663
  Test MAE: 24.0494
  Test R-squared: -0.0054

Pipeline C:
  Best CV RMSE: 49.8856
  Test RMSE: 109.0413
  Test MAE: 26.2418
  Test R-squared: -0.0087

Pipeline D:
  Best CV RMSE: 51.4202
  Test RMSE: 109.6081
  Test MAE: 26.8960
  Test R-squared: -0.0192

--- Conclusion ---
The best performing model on the test set is: Pipeline B with an RMSE of 108.8663


# Export

+ Save the best performing model to a pickle file.

In [189]:
# Export
import pickle

output_model_path = 'best_wildfire_model.pkl'

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [190]:

# trying to find structure of pipeline to use for shap code
print(best_overall_model) 
print("\n--- Pipeline Steps and Names ---")
for step_name, step_object in best_overall_model.named_steps.items():
    print(f"Step Name: '{step_name}', Step Type: {type(step_object)}")



Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('poly',
                                                                   PolynomialFeatures(include_bias=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                             

In [191]:
data_transform = best_overall_model.named_steps['preprocessor'].transform(X_test)

explainer = shap.explainers.Linear(
    best_overall_model.named_steps['regressor'], 
    data_transform,
    feature_names = best_overall_model.named_steps['preprocessor'].get_feature_names_out())

shap_values = explainer(data_transform)

#Having issues with shap, not sure if I set this up correctly or if im using an outdated library since the error says an element from numpy has been deprecated.
#Not sure how to proceed, any help would be appreciated


InvalidModelError: An unknown model type was passed: <class 'sklearn.neighbors._regression.KNeighborsRegressor'>

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.